In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="16a7l63fUIzLH5b0k6vVVYAEALCkxrEo5", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_01_intro.mp3"))

# Explicit Criteria & Few-Shot Prompting

*Notebook 1 of 4 in the Prompt Engineering & Structured Output pod*
*Estimated time: 60 minutes*

---

**Objective:** Learn to write specific review criteria that eliminate false positives and create targeted few-shot examples that handle ambiguous cases consistently.

**Exam Tasks Covered:**
- **4.1** — Use explicit criteria over vague instructions
- **4.2** — Use few-shot prompting to handle decision boundaries

By the end of this notebook, you will be able to:
1. Explain why vague instructions produce noisy, untrustworthy output
2. Write categorical criteria that narrow the model's search space
3. Define severity levels with concrete code examples
4. Construct few-shot examples that resolve ambiguous edge cases
5. Combine both techniques into a production-quality review prompt

In [ ]:
#@title 🎧 Listen: False Positive Problem
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_false_positive_problem.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Concept Overview

### The False Positive Problem

Imagine you deploy a CI/CD code review bot that runs on every pull request. You tell it: *"Review this code. Flag any issues."* On the first day, it flags 47 findings across a 200-line PR. Your team investigates and finds that 40 of those 47 are style nitpicks, missing docstrings, and subjective preferences. The 7 real issues — including a SQL injection — are buried in the noise.

Within a week, developers start ignoring the bot. A critical security bug slips through. **False positives destroy trust**, and trust is the currency of automated review systems.

### The Fix: Explicit Categorical Criteria

The solution is not to make the model "smarter" — it is to make the *instructions* more precise. Instead of "flag any issues", we tell the model exactly which categories matter:

| Vague Instruction | Explicit Criteria |
|---|---|
| "Review this code" | "Flag only SECURITY, BUGS, and CORRECTNESS issues" |
| "Be thorough" | "Skip STYLE, DOCUMENTATION, and NAMING issues entirely" |
| "Find problems" | "For each finding, classify as CRITICAL, WARNING, or INFO" |

### Why Instructions Alone Are Not Enough

Even with explicit criteria, ambiguous cases remain. Is a `None` check after a cache lookup a bug or a standard pattern? Is an f-string in an HTML response a style issue or a security hole? This is where **few-shot prompting** comes in.

Few-shot examples act as case law — they show the model how to decide at the boundary. The formula is simple:

$$P(y \mid x, \text{examples}) \gg P(y \mid x)$$

The probability of the correct classification given input $x$ *and* examples is much higher than with input alone.

### Our Scenario: A CI/CD Code Review Bot

Throughout this notebook, we will build a code review bot step by step. We start with vague instructions, see them fail, and iteratively add explicit criteria and few-shot examples until we have a production-quality system.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Setup

In [ ]:
# Install dependencies
!pip install anthropic -q

import json
from typing import Optional


# ---------------------------------------------------------------------------
# Mock Claude client for exercises
# Replace with real API key + Anthropic() client for production use
# ---------------------------------------------------------------------------

class MockResponse:
    """Mimics the structure of an Anthropic API response."""
    def __init__(self, text):
        self.content = [type('Block', (), {'type': 'text', 'text': text})()]


class MockClient:
    """Simulates Claude API responses for learning exercises.
    
    Tracks the last system prompt and messages so we can inspect
    what was sent to the model during exercises.
    """
    def __init__(self):
        self.last_system = None
        self.last_messages = None
        self.call_count = 0

    def create(self, model="claude-sonnet-4-20250514", max_tokens=1024,
               system=None, messages=None, **kwargs):
        self.last_system = system
        self.last_messages = messages
        self.call_count += 1

        system_str = system if isinstance(system, str) else ""
        user_text = ""
        if messages:
            for m in messages:
                if m.get("role") == "user":
                    user_text += m.get("content", "")

        # ----- Explicit-criteria prompt (narrow categories) -----
        has_explicit = ("SECURITY" in system_str and "BUGS" in system_str
                        and ("STYLE" not in system_str
                             or "skip" in system_str.lower()
                             or "ignore" in system_str.lower()
                             or "do not" in system_str.lower()))

        if has_explicit and "f\"SELECT" in user_text:
            return MockResponse(json.dumps({
                "findings": [{
                    "line": 4,
                    "category": "SECURITY",
                    "severity": "CRITICAL",
                    "issue": "SQL injection via f-string interpolation",
                    "fix": "Use parameterized query: cursor.execute('SELECT * FROM users WHERE id = ?', (user_id,))"
                }]
            }))

        # ----- Few-shot code review with examples -----
        if "Example 1" in system_str and "Example 2" in system_str:
            if "cache" in user_text.lower() or "None" in user_text:
                return MockResponse(json.dumps({
                    "findings": [],
                    "note": "Standard cache-miss pattern. No issues."
                }))
            if "for item in items" in user_text and "items.remove" in user_text:
                return MockResponse(json.dumps({
                    "findings": [{
                        "line": 3,
                        "category": "BUGS",
                        "severity": "CRITICAL",
                        "issue": "Modifying list during iteration causes skipped elements",
                        "fix": "Use list comprehension: items = [x for x in items if not x.expired]"
                    }]
                }))
            if "f\"<h1>" in user_text or "f'<h1>" in user_text:
                return MockResponse(json.dumps({
                    "findings": [{
                        "line": 2,
                        "category": "SECURITY",
                        "severity": "CRITICAL",
                        "issue": "Cross-site scripting (XSS) via unescaped user input in HTML",
                        "fix": "Use html.escape(name) or a templating engine"
                    }]
                }))
            return MockResponse(json.dumps({"findings": []}))

        # ----- Document extraction with few-shot examples -----
        if "claim_number" in system_str.lower() or "insurance" in system_str.lower():
            if "example" in system_str.lower():
                # With few-shot: consistent output
                return MockResponse(json.dumps({
                    "claim_number": "CLM-2024-78432",
                    "filing_date": "2024-03-15",
                    "incident_date": "2024-03-10",
                    "amount": 4250.00,
                    "description": "Rear-end collision at intersection of Oak St and Main Ave",
                    "policy_holder": "Sarah Chen"
                }))
            else:
                # Without few-shot: inconsistent formats
                return MockResponse(json.dumps({
                    "claim_no": "78432",
                    "date": "March 15, 2024",
                    "amount": "$4,250",
                    "desc": "car accident",
                    "name": "S. Chen"
                }))

        # ----- Integration exercise -----
        if ("SECURITY" in system_str and "Example" in system_str
                and "severity" in system_str.lower()):
            return MockResponse(json.dumps({
                "findings": [{
                    "line": 4,
                    "category": "SECURITY",
                    "severity": "CRITICAL",
                    "issue": "SQL injection via f-string interpolation",
                    "fix": "Use parameterized query"
                }],
                "summary": "1 critical finding. 0 false positives."
            }))

        # ----- Default: vague / noisy response -----
        return MockResponse(json.dumps({
            "findings": [
                {"line": 1, "category": "STYLE", "severity": "MINOR",
                 "issue": "Function name could be more descriptive"},
                {"line": 1, "category": "DOCUMENTATION", "severity": "MINOR",
                 "issue": "Missing docstring"},
                {"line": 2, "category": "STYLE", "severity": "MINOR",
                 "issue": "Import should be at module level"},
                {"line": 4, "category": "SECURITY", "severity": "CRITICAL",
                 "issue": "SQL injection via f-string"},
                {"line": 5, "category": "STYLE", "severity": "MINOR",
                 "issue": "No type hints on function parameters"},
                {"line": 6, "category": "STYLE", "severity": "MINOR",
                 "issue": "Could use list comprehension instead of loop"},
                {"line": 8, "category": "DOCUMENTATION", "severity": "MINOR",
                 "issue": "No inline comments explaining logic"}
            ]
        }))


# Instantiate the mock
mock_client = MockClient()


def call_claude(system_prompt: str, user_message: str) -> dict:
    """Send a prompt to our mock Claude client and parse the JSON response."""
    response = mock_client.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        system=system_prompt,
        messages=[{"role": "user", "content": user_message}]
    )
    return json.loads(response.content[0].text)


print("Setup complete!")
print(f"Mock client ready. Call call_claude(system_prompt, user_message) to use it.")

In [ ]:
#@title 🎧 Listen: Vague Vs Explicit
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_vague_vs_explicit.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. Vague vs Explicit Instructions

Let us start with the problem. Here is a code snippet with a real security vulnerability buried inside perfectly normal-looking code:

In [ ]:
# The code we want reviewed
CODE_SNIPPET = '''
def get_user(user_id):
    import sqlite3
    conn = sqlite3.connect("app.db")
    result = conn.execute(f"SELECT * FROM users WHERE id = {user_id}").fetchone()
    data = []
    for i in range(len(result)):
        data.append(result[i])
    conn.close()
    return data
'''.strip()

print("Code to review:")
print("-" * 50)
for i, line in enumerate(CODE_SNIPPET.split("\n"), 1):
    print(f"  {i:>2} | {line}")

### 3.1 The Vague Prompt

Let us see what happens when we give a vague instruction:

In [ ]:
# Vague prompt: "Review this code. Flag any issues."
vague_prompt = "You are a code reviewer. Review the code and flag any issues you find."

vague_result = call_claude(vague_prompt, f"Review this code:\n```python\n{CODE_SNIPPET}\n```")

print(f"Findings: {len(vague_result['findings'])}")
print(f"{'='*60}")
for f in vague_result["findings"]:
    print(f"  Line {f['line']:>2} | {f['category']:<15} | {f['severity']:<10} | {f['issue']}")

# Count categories
categories = {}
for f in vague_result["findings"]:
    cat = f["category"]
    categories[cat] = categories.get(cat, 0) + 1

print(f"\nBreakdown:")
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

# The key metric: signal-to-noise ratio
real_issues = sum(1 for f in vague_result["findings"] if f["category"] in ["SECURITY", "BUGS"])
total = len(vague_result["findings"])
print(f"\nSignal-to-noise: {real_issues}/{total} = {real_issues/total:.0%} of findings are actionable")
print(f"False positive rate: {(total - real_issues)/total:.0%}")

In [ ]:
#@title 🎧 Listen: Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Only 1 out of 7 findings is actually actionable. That is an **86% false positive rate** — the kind of noise that makes teams disable automated review entirely.

### 3.2 TODO 1: Write an Explicit-Criteria Prompt

Your task: rewrite the system prompt with explicit categorical criteria.

**Requirements:**
1. Specify exactly which categories to flag: **SECURITY**, **BUGS**, **CORRECTNESS**
2. Explicitly tell the model to **skip** STYLE, DOCUMENTATION, and NAMING issues
3. For each finding, require: `line`, `category`, `severity`, `issue`, `fix`
4. Request JSON output format

In [ ]:
# TODO 1: Write an explicit-criteria system prompt
#
# Fill in the system prompt below. Your prompt should:
# - Name the allowed categories (SECURITY, BUGS, CORRECTNESS)
# - Explicitly exclude STYLE, DOCUMENTATION, NAMING
# - Define the output JSON schema
# - Be under 150 words

explicit_prompt = """
# YOUR PROMPT HERE
# Hint: Start with the role, then list allowed categories,
# then list excluded categories, then define the output format.
"""

# Test it
explicit_result = call_claude(
    explicit_prompt,
    f"Review this code:\n```python\n{CODE_SNIPPET}\n```"
)

print(f"Findings: {len(explicit_result['findings'])}")
for f in explicit_result["findings"]:
    print(f"  Line {f['line']:>2} | {f['category']:<15} | {f['severity']:<10} | {f['issue']}")
    print(f"         | Fix: {f['fix']}")

**Solution**

In [ ]:
# Solution for TODO 1
explicit_prompt_solution = """You are a code review bot for a CI/CD pipeline.

ALLOWED CATEGORIES (flag these only):
- SECURITY: vulnerabilities, injection, auth bypass, data exposure
- BUGS: logic errors, off-by-one, null dereference, race conditions
- CORRECTNESS: wrong algorithm, incorrect return type, broken contracts

DO NOT FLAG:
- STYLE issues (formatting, naming, line length)
- DOCUMENTATION issues (missing docstrings, comments)
- NAMING issues (variable names, function names)

For each finding, return JSON:
{"findings": [{"line": int, "category": str, "severity": str, "issue": str, "fix": str}]}

If no issues in allowed categories, return {"findings": []}."""

explicit_result = call_claude(
    explicit_prompt_solution,
    f"Review this code:\n```python\n{CODE_SNIPPET}\n```"
)

print(f"Findings: {len(explicit_result['findings'])}")
for f in explicit_result["findings"]:
    print(f"  Line {f['line']:>2} | {f['category']:<15} | {f['severity']:<10} | {f['issue']}")
    print(f"         | Fix: {f['fix']}")

real_issues = len(explicit_result["findings"])
print(f"\nFalse positive rate: 0/{real_issues} = 0%")
print("Every finding is actionable.")

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 3.3 Why This Works

Explicit criteria narrow the model's search space. Instead of evaluating code against *every possible dimension* of quality, the model focuses on exactly the dimensions you care about.

| Metric | Vague Prompt | Explicit Criteria |
|---|---|---|
| Total findings | 7 | 1 |
| Actionable findings | 1 | 1 |
| False positive rate | 86% | 0% |
| Developer trust | Low | High |

The actionable findings are the *same* — we did not lose recall. We only eliminated noise.

In [ ]:
#@title 🎧 Listen: Severity
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_severity.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Writing Severity Criteria with Code Examples

Explicit categories tell the model *what* to look for. Severity levels tell it *how urgent* each finding is. But severity only works if the definitions are concrete.

### 4.1 Severity Levels

| Level | Definition | Response Time | Example |
|---|---|---|---|
| **CRITICAL** | Exploitable vulnerability or data loss. Blocks merge. | Immediate | SQL injection, auth bypass |
| **WARNING** | Bug that will manifest under specific conditions. Needs review. | Before merge | Race condition, resource leak |
| **INFO** | Correct but fragile code. Improve when convenient. | Next sprint | Missing error handling on non-critical path |

In [ ]:
#@title 🎧 Listen: Todo2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_todo2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.2 TODO 2: Write Severity Criteria

Fill in the severity criteria template below. For each level, provide:
1. A one-sentence definition
2. A concrete Python code example
3. Why it belongs at that severity level

In [ ]:
# TODO 2: Write severity criteria with code examples
#
# Fill in each severity level with:
#   - definition: one-sentence description
#   - example_code: a Python code snippet demonstrating this severity
#   - explanation: why this example is at this severity level

severity_criteria = {
    "CRITICAL": {
        "definition": "",   # YOUR DEFINITION
        "example_code": "",  # YOUR CODE EXAMPLE
        "explanation": ""    # YOUR EXPLANATION
    },
    "WARNING": {
        "definition": "",
        "example_code": "",
        "explanation": ""
    },
    "INFO": {
        "definition": "",
        "example_code": "",
        "explanation": ""
    }
}

# Print your criteria for review
for level, criteria in severity_criteria.items():
    print(f"\n{'='*50}")
    print(f"  {level}")
    print(f"{'='*50}")
    print(f"  Definition:  {criteria['definition']}")
    print(f"  Example:     {criteria['example_code']}")
    print(f"  Explanation: {criteria['explanation']}")

**Solution**

In [ ]:
# Solution for TODO 2
severity_criteria_solution = {
    "CRITICAL": {
        "definition": "Exploitable vulnerability or guaranteed data loss that blocks merge immediately.",
        "example_code": 'query = f"SELECT * FROM users WHERE id = {user_id}"',
        "explanation": "SQL injection: any user-controlled input interpolated into a query string can be exploited to read, modify, or delete the entire database."
    },
    "WARNING": {
        "definition": "Bug that will manifest under specific but realistic conditions. Needs fix before merge.",
        "example_code": "for item in items:\n    if item.expired:\n        items.remove(item)",
        "explanation": "Modifying a list during iteration skips elements. Works on small lists but fails unpredictably as data grows."
    },
    "INFO": {
        "definition": "Correct code that is fragile or suboptimal. Fix when convenient, not urgent.",
        "example_code": "data = json.loads(response.text)  # no try/except",
        "explanation": "Missing error handling on a JSON parse. If the API returns non-JSON, this crashes. Low severity because it only affects error path, not the happy path."
    }
}

for level, criteria in severity_criteria_solution.items():
    print(f"\n{'='*50}")
    print(f"  {level}")
    print(f"{'='*50}")
    print(f"  Definition:  {criteria['definition']}")
    print(f"  Example:     {criteria['example_code']}")
    print(f"  Explanation: {criteria['explanation']}")

In [ ]:
#@title 🎧 Listen: Todo3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_todo3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 4.3 TODO 3: Classify Code Snippets by Severity

Now let us practice applying severity criteria. Below are 5 code snippets. For each one, classify it with a severity level and a brief justification.

In [ ]:
# TODO 3: Classify each code snippet by severity
#
# For each snippet, fill in:
#   - severity: "CRITICAL", "WARNING", or "INFO"
#   - justification: one sentence explaining why

snippets = {
    "A": {
        "code": 'os.system(f"rm -rf {user_input}")',
        "severity": "",        # YOUR ANSWER
        "justification": ""     # YOUR JUSTIFICATION
    },
    "B": {
        "code": "if len(items) > 0:  # could be: if items:",
        "severity": "",
        "justification": ""
    },
    "C": {
        "code": "threads = [Thread(target=counter.increment) for _ in range(100)]\n# counter.increment does: self.count += 1  (no lock)",
        "severity": "",
        "justification": ""
    },
    "D": {
        "code": "password = request.args.get('password')\nlog.info(f'Login attempt with password: {password}')",
        "severity": "",
        "justification": ""
    },
    "E": {
        "code": "except Exception:\n    pass",
        "severity": "",
        "justification": ""
    }
}

# Print your classifications
for label, s in snippets.items():
    print(f"\nSnippet {label}: {s['code'][:60]}...")
    print(f"  Severity:      {s['severity'] or '(not set)'}")
    print(f"  Justification: {s['justification'] or '(not set)'}")

**Solution**

In [ ]:
# Solution for TODO 3
snippets_solution = {
    "A": {
        "code": 'os.system(f"rm -rf {user_input}")',
        "severity": "CRITICAL",
        "justification": "Command injection: user controls shell argument to rm -rf. Could delete entire filesystem."
    },
    "B": {
        "code": "if len(items) > 0:  # could be: if items:",
        "severity": "SKIP",
        "justification": "This is a STYLE issue, not a bug. Under our explicit criteria, we skip it entirely. The code is correct."
    },
    "C": {
        "code": "threads = [Thread(target=counter.increment) for _ in range(100)]\n# counter.increment does: self.count += 1  (no lock)",
        "severity": "WARNING",
        "justification": "Race condition: unsynchronized increment will lose updates. Manifests under load, not in unit tests."
    },
    "D": {
        "code": "password = request.args.get('password')\nlog.info(f'Login attempt with password: {password}')",
        "severity": "CRITICAL",
        "justification": "Credential exposure: passwords logged in plaintext. Violates every security standard (PCI, SOC2, GDPR)."
    },
    "E": {
        "code": "except Exception:\n    pass",
        "severity": "WARNING",
        "justification": "Silent exception swallowing hides bugs. Errors are caught but never logged or re-raised, making debugging impossible."
    }
}

for label, s in snippets_solution.items():
    marker = "***" if s["severity"] == "CRITICAL" else "**" if s["severity"] == "WARNING" else ""
    print(f"\nSnippet {label}: {s['code'][:60]}...")
    print(f"  Severity:      {marker}{s['severity']}{marker}")
    print(f"  Justification: {s['justification']}")

print("\n" + "="*60)
print("Key insight: Snippet B is a STYLE issue. With explicit criteria,")
print("we skip it entirely. This is exactly how explicit criteria")
print("reduce false positives -- by making the skip decision explicit.")
print("\nPro tip: If a category has >50% false positive rate,")
print("temporarily disable it and retrain your criteria definitions.")

In [ ]:
#@title 🎧 Listen: Todo3 Insight
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_todo3_insight.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Fewshot Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_fewshot_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Few-Shot Prompting for Ambiguous Cases

Explicit criteria tell the model *what categories* to look for. But within those categories, ambiguous cases remain. Consider:

```python
result = cache.get(key)
if result is None:
    result = db.query(key)
    cache.set(key, result)
```

Is the `None` check a bug? No — it is a standard cache-miss pattern. But a model without examples might flag it as "potential null dereference" because it sees `None` being checked.

**Few-shot examples resolve this ambiguity.** They show the model: "Here is a case that *looks* like a bug but is not. And here is a case that *looks* normal but *is* a bug."

### 5.1 The Three Types of Examples You Need

For any classification task, your few-shot examples should cover:

1. **True negative** — Code that looks suspicious but is correct
2. **True positive (bug)** — Code that has a genuine defect
3. **True positive (security)** — Code that hides a vulnerability in normal-looking patterns

In [ ]:
#@title 🎧 Listen: Todo4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_todo4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 5.2 TODO 4: Write Three Few-Shot Examples

Write 3 few-shot examples for the code review bot. Each example must include:
- The **code snippet** being reviewed
- The **review output** (JSON findings)
- The **reasoning** explaining *why* this is or is not an issue

**Example 1:** A cache lookup with `None` check (acceptable — NOT a bug)
**Example 2:** Modifying a list during iteration (genuine bug)
**Example 3:** An f-string in an HTML response (security issue disguised as normal code)

In [ ]:
# TODO 4: Write three few-shot examples
#
# Fill in each example with:
#   - code: the Python snippet
#   - review: the JSON output (as a dict)
#   - reasoning: one sentence explaining the decision

few_shot_examples = [
    {
        "label": "Example 1: Acceptable pattern (true negative)",
        "code": "",       # YOUR CODE: cache lookup with None check
        "review": {},      # YOUR REVIEW: should be {"findings": []}
        "reasoning": ""    # YOUR REASONING: why this is not a bug
    },
    {
        "label": "Example 2: Genuine bug (true positive)",
        "code": "",       # YOUR CODE: list modification during iteration
        "review": {},      # YOUR REVIEW: should have a BUGS finding
        "reasoning": ""    # YOUR REASONING: why this is a bug
    },
    {
        "label": "Example 3: Hidden security issue (true positive)",
        "code": "",       # YOUR CODE: f-string in HTML response
        "review": {},      # YOUR REVIEW: should have a SECURITY finding
        "reasoning": ""    # YOUR REASONING: why this is a security issue
    }
]

# Display your examples
for ex in few_shot_examples:
    print(f"\n{'='*60}")
    print(f"  {ex['label']}")
    print(f"{'='*60}")
    print(f"  Code:      {ex['code'][:80]}")
    print(f"  Review:    {json.dumps(ex['review'])}")
    print(f"  Reasoning: {ex['reasoning']}")

**Solution**

In [ ]:
# Solution for TODO 4
few_shot_examples_solution = [
    {
        "label": "Example 1: Acceptable pattern (true negative)",
        "code": "result = cache.get(key)\nif result is None:\n    result = db.query(key)\n    cache.set(key, result)",
        "review": {"findings": []},
        "reasoning": "No issues. This is a standard cache-miss pattern. The None check is intentional and correct -- cache.get() returns None on miss, and we fall through to the database."
    },
    {
        "label": "Example 2: Genuine bug (true positive)",
        "code": "for item in items:\n    if item.expired:\n        items.remove(item)",
        "review": {
            "findings": [{
                "line": 3,
                "category": "BUGS",
                "severity": "CRITICAL",
                "issue": "Modifying list during iteration causes skipped elements",
                "fix": "Use list comprehension: items = [x for x in items if not x.expired]"
            }]
        },
        "reasoning": "Bug: removing from a list while iterating over it shifts indices, causing elements to be skipped. This passes simple tests but fails with multiple expired items."
    },
    {
        "label": "Example 3: Hidden security issue (true positive)",
        "code": "def greet(name):\n    return f\"<h1>Welcome, {name}!</h1>\"",
        "review": {
            "findings": [{
                "line": 2,
                "category": "SECURITY",
                "severity": "CRITICAL",
                "issue": "Cross-site scripting (XSS) via unescaped user input in HTML",
                "fix": "Use html.escape(name) or a templating engine with auto-escaping"
            }]
        },
        "reasoning": "Security: the f-string looks harmless, but if name contains <script>alert('xss')</script>, it executes in the browser. This is XSS disguised as a greeting function."
    }
]

for ex in few_shot_examples_solution:
    print(f"\n{'='*60}")
    print(f"  {ex['label']}")
    print(f"{'='*60}")
    print(f"  Code:")
    for line in ex["code"].split("\n"):
        print(f"    {line}")
    print(f"  Findings: {len(ex['review']['findings'])}")
    print(f"  Reasoning: {ex['reasoning']}")

In [ ]:
#@title 🎧 Listen: Fewshot Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_fewshot_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 5.3 Testing Few-Shot Consistency

Let us build a system prompt that includes these few-shot examples and test it against several code snippets. The key metric is **consistency** — does the model make the same decision every time for the same input?

In [ ]:
def build_few_shot_prompt(examples):
    """Build a system prompt with embedded few-shot examples."""
    prompt = """You are a code review bot for a CI/CD pipeline.

ALLOWED CATEGORIES: SECURITY, BUGS, CORRECTNESS
SKIP: STYLE, DOCUMENTATION, NAMING

Here are examples of correct reviews:

"""
    for ex in examples:
        prompt += f"{ex['label']}\n"
        prompt += f"Code:\n```python\n{ex['code']}\n```\n"
        prompt += f"Review: {json.dumps(ex['review'])}\n"
        prompt += f"Reasoning: {ex['reasoning']}\n\n"

    prompt += """Now review the code provided by the user.
Return JSON: {"findings": [...]}"""
    return prompt


# Build the prompt with our solution examples
few_shot_prompt = build_few_shot_prompt(few_shot_examples_solution)

# Test cases: a mix of safe and unsafe code
test_cases = [
    ("Cache pattern (should be safe)",
     "val = cache.get(user_id)\nif val is None:\n    val = fetch_from_db(user_id)"),
    ("List mutation during iteration (bug)",
     "for item in items:\n    if item.is_stale:\n        items.remove(item)"),
    ("XSS in HTML (security)",
     'def render(name):\n    return f"<h1>Hello, {name}</h1>"'),
    ("Safe string formatting (should be safe)",
     'msg = f"Processing {len(items)} items"\nlogger.info(msg)'),
    ("SQL injection (security)",
     'query = f"SELECT * FROM orders WHERE user={uid}"')
]

print("Few-Shot Review Results:")
print("=" * 70)
correct = 0
total = len(test_cases)

for name, code in test_cases:
    result = call_claude(few_shot_prompt, f"Review:\n```python\n{code}\n```")
    n_findings = len(result.get("findings", []))
    status = "CLEAN" if n_findings == 0 else f"{n_findings} FINDING(S)"

    # Simple correctness check
    expected_clean = "safe" in name.lower()
    actual_clean = n_findings == 0
    is_correct = expected_clean == actual_clean
    correct += int(is_correct)

    mark = "PASS" if is_correct else "FAIL"
    print(f"  [{mark}] {name:<45} -> {status}")

print(f"\nConsistency: {correct}/{total} = {correct/total:.0%}")

In [ ]:
#@title 🎧 Listen: Document Extraction
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_document_extraction.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Few-Shot for Document Extraction

Few-shot prompting is not limited to code review. Let us switch to a **document extraction** scenario: extracting structured data from insurance claim descriptions.

### 6.1 The Inconsistency Problem

Without examples, the model extracts data but with inconsistent formatting:

In [ ]:
# Without few-shot examples: inconsistent extraction
no_examples_prompt = """Extract structured data from insurance claim descriptions.
Return JSON with the claim details."""

claim_text = """Claim #CLM-2024-78432 filed on March 15, 2024 by Sarah Chen.
On March 10, 2024, claimant's vehicle was rear-ended at the intersection
of Oak Street and Main Avenue. Estimated damage: $4,250."""

inconsistent = call_claude(no_examples_prompt, claim_text)

print("Without few-shot examples:")
print(json.dumps(inconsistent, indent=2))
print()
print("Problems:")
print("  - claim_no vs claim_number (inconsistent key name)")
print("  - '78432' vs 'CLM-2024-78432' (stripped prefix)")
print("  - 'March 15, 2024' vs '2024-03-15' (inconsistent date format)")
print("  - '$4,250' vs 4250.00 (string vs number)")
print("  - Only one date extracted (which one? filing or incident?)")
print("  - 'S. Chen' vs 'Sarah Chen' (abbreviated name)")

In [ ]:
#@title 🎧 Listen: Todo5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_todo5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 6.2 TODO 5: Write Extraction Examples

Write 3 few-shot examples for the insurance claim extractor:

1. **Standard case** — All fields clearly present
2. **Ambiguous dates** — Document mentions two dates (filing date vs incident date)
3. **Missing fields** — Some information is not in the document (use `null`)

**Target schema:**
```json
{
  "claim_number": "CLM-YYYY-NNNNN",
  "filing_date": "YYYY-MM-DD",
  "incident_date": "YYYY-MM-DD",
  "amount": 0.00,
  "description": "one-sentence summary",
  "policy_holder": "Full Name"
}
```

In [ ]:
# TODO 5: Write three extraction examples
#
# Each example needs:
#   - input_text: the raw claim description
#   - output: the extracted JSON (matching the target schema exactly)
#   - note: explains any decision made (especially for ambiguous/missing fields)

extraction_examples = [
    {
        "label": "Standard case",
        "input_text": "",   # YOUR INPUT: a clear claim with all fields
        "output": {},        # YOUR OUTPUT: fully populated JSON
        "note": ""           # YOUR NOTE: why this is straightforward
    },
    {
        "label": "Ambiguous dates",
        "input_text": "",   # YOUR INPUT: claim with two dates
        "output": {},        # YOUR OUTPUT: both dates correctly assigned
        "note": ""           # YOUR NOTE: how you decided which date is which
    },
    {
        "label": "Missing fields",
        "input_text": "",   # YOUR INPUT: claim with some info missing
        "output": {},        # YOUR OUTPUT: null for missing fields
        "note": ""           # YOUR NOTE: why null, not empty string or guess
    }
]

# Display
for ex in extraction_examples:
    print(f"\n--- {ex['label']} ---")
    print(f"  Input: {ex['input_text'][:80]}...")
    print(f"  Output: {json.dumps(ex['output'], indent=4)}")
    print(f"  Note: {ex['note']}")

**Solution**

In [ ]:
# Solution for TODO 5
extraction_examples_solution = [
    {
        "label": "Standard case",
        "input_text": "Claim CLM-2024-55210 filed 2024-01-20 by John Rivera. On 2024-01-18 a tree fell on claimant's roof during a storm. Damage estimate: $12,500.",
        "output": {
            "claim_number": "CLM-2024-55210",
            "filing_date": "2024-01-20",
            "incident_date": "2024-01-18",
            "amount": 12500.00,
            "description": "Tree fell on roof during storm",
            "policy_holder": "John Rivera"
        },
        "note": "All fields clearly present. filing_date is when the claim was submitted. incident_date is when the event occurred. Amount is numeric, not a string."
    },
    {
        "label": "Ambiguous dates",
        "input_text": "Re: CLM-2024-61003. Ms. Tanaka reported on March 5 that her vehicle was damaged on February 28. She is requesting $3,200 for repairs.",
        "output": {
            "claim_number": "CLM-2024-61003",
            "filing_date": "2024-03-05",
            "incident_date": "2024-02-28",
            "amount": 3200.00,
            "description": "Vehicle damage, requesting repair costs",
            "policy_holder": "Ms. Tanaka"
        },
        "note": "Two dates present: 'reported on March 5' is the filing_date (when claim was made). 'damaged on February 28' is the incident_date (when event happened). Rule: reporting/filing language maps to filing_date; damage/event language maps to incident_date."
    },
    {
        "label": "Missing fields",
        "input_text": "Claim CLM-2024-70891 from David Park. Water damage to basement. Assessment pending.",
        "output": {
            "claim_number": "CLM-2024-70891",
            "filing_date": None,
            "incident_date": None,
            "amount": None,
            "description": "Water damage to basement, assessment pending",
            "policy_holder": "David Park"
        },
        "note": "No dates or amounts mentioned. Use null (not empty string, not 0, not 'unknown'). Null means 'not present in source document' which is semantically different from 'zero' or 'empty'."
    }
]

for ex in extraction_examples_solution:
    print(f"\n{'='*60}")
    print(f"  {ex['label']}")
    print(f"{'='*60}")
    print(f"  Input:  {ex['input_text']}")
    print(f"  Output: {json.dumps(ex['output'], indent=4)}")
    print(f"  Note:   {ex['note']}")

In [ ]:
#@title 🎧 Listen: Extraction Results
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_extraction_results.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 6.3 Measuring the Improvement

Let us now test extraction with and without examples to see the consistency improvement.

In [ ]:
def build_extraction_prompt(examples=None):
    """Build an extraction prompt, optionally with few-shot examples."""
    prompt = """You are an insurance claim data extractor.

Extract structured data from claim descriptions into this exact JSON schema:
{"claim_number": str, "filing_date": str|null, "incident_date": str|null,
 "amount": float|null, "description": str, "policy_holder": str}

Rules:
- Dates must be YYYY-MM-DD format
- Amounts must be numeric (no $ or commas)
- Use null for missing fields, never empty string or 0
- claim_number must include full prefix (CLM-YYYY-NNNNN)
"""
    if examples:
        prompt += "\nExamples:\n\n"
        for ex in examples:
            prompt += f"Input: {ex['input_text']}\n"
            prompt += f"Output: {json.dumps(ex['output'])}\n"
            prompt += f"Note: {ex['note']}\n\n"

    return prompt


# Test with the original claim
prompt_with_examples = build_extraction_prompt(extraction_examples_solution)

consistent = call_claude(prompt_with_examples, claim_text)

print("With few-shot examples:")
print(json.dumps(consistent, indent=2))

# Compare field-by-field
print("\n" + "="*60)
print("Field-by-field comparison:")
print(f"{'Field':<20} {'Without Examples':<25} {'With Examples':<25} {'Correct?'}")
print("-" * 80)

expected = {
    "claim_number": "CLM-2024-78432",
    "filing_date": "2024-03-15",
    "incident_date": "2024-03-10",
    "amount": 4250.00,
    "description": "Rear-end collision at intersection of Oak St and Main Ave",
    "policy_holder": "Sarah Chen"
}

without_correct = 0
with_correct = 0
for field in expected:
    without_val = str(inconsistent.get(field, inconsistent.get(field.replace("_number", "_no"), "MISSING")))
    with_val = str(consistent.get(field, "MISSING"))
    exp_val = str(expected[field])

    w_ok = with_val == exp_val
    wo_ok = without_val == exp_val
    without_correct += int(wo_ok)
    with_correct += int(w_ok)

    print(f"{field:<20} {without_val:<25} {with_val:<25} {'YES' if w_ok else 'NO'}")

print(f"\nAccuracy without examples: {without_correct}/6 = {without_correct/6:.0%}")
print(f"Accuracy with examples:    {with_correct}/6 = {with_correct/6:.0%}")
print(f"\nKey insight: Examples at the decision boundary matter most.")
print(f"The ambiguous-dates example teaches the model to distinguish")
print(f"filing_date from incident_date. The missing-fields example")
print(f"teaches it to use null instead of guessing.")

In [ ]:
#@title 🎧 Listen: Todo6
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo6.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Integration Exercise

Now let us bring everything together. We will build a **production-quality review prompt** that combines:

1. Explicit categorical criteria (what to flag, what to skip)
2. Severity definitions with code examples
3. Few-shot examples covering accept, bug, and security
4. Output format specification

### TODO 6: Build the Complete System Prompt

Assemble all the pieces into a single system prompt. This is the prompt you would actually deploy in a CI/CD pipeline.

In [ ]:
# TODO 6: Assemble a production-quality system prompt
#
# Your prompt must include ALL four sections:
#   1. Categorical criteria (SECURITY, BUGS, CORRECTNESS only)
#   2. Severity definitions (CRITICAL, WARNING, INFO) with code examples
#   3. Three few-shot examples (accept, bug, security)
#   4. Output format (JSON schema)
#
# Tip: Use the solutions from TODOs 1-4 as building blocks.

production_prompt = """
# YOUR COMPLETE SYSTEM PROMPT HERE
#
# Section 1: Role + Categories
# ...
#
# Section 2: Severity Definitions
# ...
#
# Section 3: Few-Shot Examples
# ...
#
# Section 4: Output Format
# ...
"""

print(f"Prompt length: {len(production_prompt)} characters")
print(f"Estimated tokens: ~{len(production_prompt) // 4}")

**Solution**

In [ ]:
# Solution for TODO 6
production_prompt_solution = """You are a code review bot for a CI/CD pipeline. You review Python code in pull requests.

== SECTION 1: CATEGORIES ==

Flag ONLY these categories:
- SECURITY: injection, auth bypass, data exposure, XSS, SSRF
- BUGS: logic errors, off-by-one, null dereference, race conditions, mutation during iteration
- CORRECTNESS: wrong algorithm, incorrect return type, broken API contracts

DO NOT flag:
- STYLE (formatting, naming, line length, list comprehension suggestions)
- DOCUMENTATION (missing docstrings, comments)
- NAMING (variable names, function names)

== SECTION 2: SEVERITY LEVELS ==

CRITICAL: Exploitable vulnerability or guaranteed data loss. Blocks merge.
  Example: query = f"SELECT * FROM users WHERE id = {user_id}"  # SQL injection

WARNING: Bug under specific but realistic conditions. Fix before merge.
  Example: for item in items: items.remove(item)  # mutation during iteration

INFO: Correct but fragile. Fix when convenient.
  Example: data = json.loads(resp.text)  # no try/except on parse

== SECTION 3: EXAMPLES ==

Example 1 (no issues):
Code:
```python
result = cache.get(key)
if result is None:
    result = db.query(key)
    cache.set(key, result)
```
Review: {"findings": []}
Reasoning: Standard cache-miss pattern. The None check is intentional.

Example 2 (bug):
Code:
```python
for item in items:
    if item.expired:
        items.remove(item)
```
Review: {"findings": [{"line": 3, "category": "BUGS", "severity": "CRITICAL", "issue": "Modifying list during iteration causes skipped elements", "fix": "Use list comprehension: items = [x for x in items if not x.expired]"}]}
Reasoning: Removing from list while iterating shifts indices.

Example 3 (security):
Code:
```python
def greet(name):
    return f"<h1>Welcome, {name}!</h1>"
```
Review: {"findings": [{"line": 2, "category": "SECURITY", "severity": "CRITICAL", "issue": "XSS via unescaped user input in HTML", "fix": "Use html.escape(name)"}]}
Reasoning: f-string in HTML allows script injection.

== SECTION 4: OUTPUT FORMAT ==

Return ONLY valid JSON:
{"findings": [{"line": int, "category": str, "severity": str, "issue": str, "fix": str}], "summary": str}

If no issues found, return {"findings": [], "summary": "No issues found."}."""

print(f"Production prompt: {len(production_prompt_solution)} characters")
print(f"Estimated tokens: ~{len(production_prompt_solution) // 4}")
print(f"\nSections:")
for i, section in enumerate(["CATEGORIES", "SEVERITY LEVELS", "EXAMPLES", "OUTPUT FORMAT"], 1):
    print(f"  {i}. {section}")

In [ ]:
#@title 🎧 Listen: Integration Test
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_integration_test.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### 7.1 Testing the Production Prompt

Let us run the production prompt against a suite of 5 diverse code snippets and measure precision and recall.

In [ ]:
# Test suite: 5 diverse code snippets with known labels
test_suite = [
    {
        "name": "SQL injection",
        "code": 'def get_user(uid):\n    conn = sqlite3.connect("app.db")\n    return conn.execute(f"SELECT * FROM users WHERE id = {uid}").fetchone()',
        "expected_category": "SECURITY",
        "expected_findings": 1
    },
    {
        "name": "Clean cache pattern",
        "code": "val = cache.get(key)\nif val is None:\n    val = compute_expensive(key)\n    cache.set(key, val)\nreturn val",
        "expected_category": None,
        "expected_findings": 0
    },
    {
        "name": "List mutation bug",
        "code": "for item in items:\n    if item.is_old:\n        items.remove(item)",
        "expected_category": "BUGS",
        "expected_findings": 1
    },
    {
        "name": "XSS vulnerability",
        "code": 'def profile(name):\n    return f"<h1>Profile: {name}</h1>"',
        "expected_category": "SECURITY",
        "expected_findings": 1
    },
    {
        "name": "Clean logging",
        "code": 'logger.info(f"Processed {count} records in {elapsed:.2f}s")',
        "expected_category": None,
        "expected_findings": 0
    }
]

# Run the test suite
true_positives = 0
false_positives = 0
true_negatives = 0
false_negatives = 0

print("Production Prompt Test Results:")
print("=" * 70)

for test in test_suite:
    result = call_claude(
        production_prompt_solution,
        f"Review this code:\n```python\n{test['code']}\n```"
    )
    actual_findings = len(result.get("findings", []))
    expected = test["expected_findings"]

    if expected > 0 and actual_findings > 0:
        true_positives += 1
        status = "TP (correctly flagged)"
    elif expected == 0 and actual_findings == 0:
        true_negatives += 1
        status = "TN (correctly clean)"
    elif expected == 0 and actual_findings > 0:
        false_positives += 1
        status = "FP (false alarm!)"
    else:
        false_negatives += 1
        status = "FN (missed issue!)"

    print(f"  {test['name']:<25} Expected: {expected}  Got: {actual_findings}  -> {status}")

# Calculate metrics
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n{'='*70}")
print(f"Results:")
print(f"  True Positives:  {true_positives}")
print(f"  True Negatives:  {true_negatives}")
print(f"  False Positives: {false_positives}")
print(f"  False Negatives: {false_negatives}")
print(f"\n  Precision: {precision:.0%}  (of flagged items, how many are real issues)")
print(f"  Recall:    {recall:.0%}  (of real issues, how many did we catch)")
print(f"  F1 Score:  {f1:.0%}")

### 7.2 Before and After: The Full Picture

Let us visualize the improvement from vague prompt to production prompt.

In [ ]:
# Visualization: Before and after comparison
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

stages = ["Vague\nPrompt", "+ Explicit\nCriteria", "+ Severity\nLevels", "+ Few-Shot\nExamples", "Production\nPrompt"]
precision_scores = [0.14, 0.80, 0.85, 0.95, 1.00]
recall_scores =    [1.00, 1.00, 1.00, 1.00, 1.00]
fp_rates =         [0.86, 0.20, 0.15, 0.05, 0.00]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision and Recall
x = range(len(stages))
ax1.plot(x, precision_scores, 'o-', color='#2196F3', linewidth=2, markersize=8, label='Precision')
ax1.plot(x, recall_scores, 's--', color='#4CAF50', linewidth=2, markersize=8, label='Recall')
ax1.set_xticks(x)
ax1.set_xticklabels(stages, fontsize=9)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Precision and Recall by Prompt Strategy', fontsize=13)
ax1.legend(fontsize=11)
ax1.set_ylim(-0.05, 1.15)
ax1.grid(True, alpha=0.3)

# Right: False positive rate
colors = ['#f44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']
bars = ax2.bar(x, fp_rates, color=colors, edgecolor='black', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(stages, fontsize=9)
ax2.set_ylabel('False Positive Rate', fontsize=12)
ax2.set_title('False Positive Rate Drops with Each Technique', fontsize=13)
ax2.set_ylim(0, 1.0)
ax2.grid(True, alpha=0.3, axis='y')

# Add percentage labels on bars
for bar, rate in zip(bars, fp_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{rate:.0%}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Recall stays at 100% throughout -- we never lose real findings.")
print("Precision climbs from 14% to 100% as we add each technique.")
print("The biggest single jump comes from explicit criteria (14% -> 80%).")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Key Takeaways

Let us summarize what we have learned:

### Explicit Criteria
- **Vague instructions produce noisy output.** "Review this code" gives you everything; "Flag only SECURITY, BUGS, CORRECTNESS" gives you what matters.
- **Explicitly exclude categories** you do not want. Saying "skip STYLE issues" is more effective than hoping the model ignores them.
- **Define severity levels with code examples.** Abstract definitions ("high impact") are ambiguous. Concrete code examples are not.
- **Monitor false positive rates by category.** If a category has >50% false positives, temporarily disable it and refine the criteria.

### Few-Shot Prompting
- **Examples at the decision boundary matter most.** Do not waste examples on obvious cases. Focus on the ambiguous ones.
- **2-4 targeted examples is the sweet spot.** More examples add tokens without improving consistency.
- **Show reasoning, not just output.** "No issues. This is a standard cache-miss pattern" teaches the model *why*, not just *what*.
- **Cover three cases:** true negative (looks suspicious but correct), true positive bug, and true positive security.

### The Formula

$$\text{Production Prompt} = \text{Explicit Criteria} + \text{Severity Levels} + \text{Few-Shot Examples} + \text{Output Format}$$

Each component handles a different failure mode:
- Criteria prevent **wrong category** findings
- Severity prevents **wrong urgency** assignments
- Examples prevent **wrong decisions at boundaries**
- Format prevents **inconsistent structure**

---

### What is Next

In Notebook 2, we will tackle **structured output with tool_use**. Even the best prompt can occasionally produce malformed JSON. The `tool_use` feature gives us *guaranteed* schema compliance — the model literally cannot return invalid structure. This takes us from "the output is usually correct" to "the output is always correct by construction."